# Scenario 4 Workbench: Developer Productivity with Claude

Welcome to the hands-on offline lab for **Scenario 4**. Read [`SCENARIO_GUIDE.md`](SCENARIO_GUIDE.md) first for the progressive discovery funnel, MCP resource rules, and 6 practice questions.

### What You Will Observe in This Lab
This notebook runs [`simulation.py`](simulation.py) offline so you can watch `DeveloperProductivityToolbox`:
1. Read static OpenAPI specs directly via an **MCP Resource** URI instead of guessing tool arguments.
2. Locate files and symbols cheaply using `Glob` and `Grep` before reading code.
3. Recover safely from an `Edit` collision by expanding the target anchor (and falling back to `Read` + `Write` when needed).

**Memory Hook:** *"Glob names, Grep content, Read last, Edit with a unique anchor."*

In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_4_developer_productivity.simulation import *

print(' Scenario Workbench Initialized: Scenario 4: Developer Productivity with Claude Code & MCP')
print(' Zero live API calls, 100% offline determinism')


### Step 2: Read MCP Resources & Search with `Grep` and `Glob`
Inspect the static Billing API spec as an MCP resource, find `def login` with `Grep`, and list test files with `Glob`.

In [ ]:
# Test MCP Resource Direct Access & Built-in Tools
files = {
    "src/auth.py": "def login(): pass",
    "src/utils.py": "def helper(): pass",
    "tests/test_auth.py": "def test_login(): pass"
}
toolbox = DeveloperProductivityToolbox(files)

# 1. MCP Resource read
mcp_res = toolbox.read_mcp_resource("resource://specs/openapi.json")
print("MCP Resource:", mcp_res["name"])
assert mcp_res["success"] is True

# 2. Grep search
grep_res = toolbox.grep_symbols("def login")
print("Grep Matches:", grep_res)
assert len(grep_res) == 1
assert grep_res[0]["file"] == "src/auth.py"

# 3. Glob paths
glob_res = toolbox.glob_paths("tests/*.py")
print("Glob Matches:", glob_res)
assert len(glob_res) == 1



### Step 3: Recover Cleanly from an `Edit` Anchor Collision
Attempt to replace `return 1` in a file where it appears twice, watch the tool catch `AMBIGUOUS_COLLISION`, and recover cleanly by expanding the anchor to include `def a(): return 1`.

In [ ]:
# Test Edit Tool Collision and Safe Fallback Protocol
calc_files = {"calc.py": "def a(): return 1\ndef b(): return 1\n"}
tb2 = DeveloperProductivityToolbox(calc_files)

# Attempt ambiguous edit
collision = tb2.apply_edit("calc.py", "return 1", "return 2")
print("Collision Result:", collision["status"])
assert collision["success"] is False
assert collision["status"] == "AMBIGUOUS_COLLISION"

# Fallback with expanded anchor
recovery = tb2.execute_edit_with_fallback(
    "calc.py",
    old_str="return 1",
    new_str="return 2",
    expanded_anchor="def a(): return 1"
)
print("Recovery Result:", recovery["final_status"])
assert recovery["final_status"] == "EXPANDED_ANCHOR_SUCCESS"



### Lab Complete
All offline checks passed. For full unit tests, run `uv run pytest scenarios/scenario_4_developer_productivity/test_scenario_4_simulation.py` in WSL Ubuntu-26.04.